In [ ]:
import joblib  # This is the tool that saves classical ML models
import os # For file handling and directory management
import cv2 # OpenCV for image processing
import numpy as np # For numerical operations and array handling
import pandas as pd # For data manipulation and results logging
import tensorflow as tf # For building and training the CNN model
from tensorflow.keras import layers, models # For defining the CNN architecture

from sklearn.model_selection import train_test_split # For splitting data into training, validation, and test sets
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score # For evaluating model performance
from sklearn.ensemble import RandomForestClassifier 


BASE_DIR = "Dataset"
IMG_SIZE = 64  
DATASET_SIZE = 5000
FILTERS = ["Raw", "Gaussian", "Median", "Bilateral", "Combined"]

results_log = []

# Apply filters to the image based on the specified filter type
def apply_filter(img, filter_type):
    if filter_type == "Raw": return img
    elif filter_type == "Gaussian": return cv2.GaussianBlur(img, (5, 5), 0) # 5x5 kernel, sigma=0 (auto)
    elif filter_type == "Median": return cv2.medianBlur(img, 5) # 5x5 kernel
    elif filter_type == "Bilateral": return cv2.bilateralFilter(img, 9, 75, 75) # Diameter=9, SigmaColor=75, SigmaSpace=75
    elif filter_type == "Combined":
        img = cv2.GaussianBlur(img, (5, 5), 0)
        img = cv2.medianBlur(img, 5)
        return cv2.bilateralFilter(img, 9, 75, 75)
    return img

# Load images, apply filters, and prepare data for ML models
def load_data(total_needed, filter_name):
    print(f"Loading {total_needed} images ({filter_name})...")
    data, labels = [], []
    per_class = total_needed // 2
    categories = ["no", "yes"] 
    
    for label_code, category in enumerate(categories):
        folder_path = os.path.join(BASE_DIR, category)
        if not os.path.exists(folder_path): 
            print(f"Error: Folder '{folder_path}' missing.")
            return np.array([]), np.array([])
            
        files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))][:per_class]
        
        for file in files:
            img = cv2.imread(os.path.join(folder_path, file), cv2.IMREAD_GRAYSCALE)
            if img is not None:
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                img = apply_filter(img, filter_name)
                data.append(img.flatten())
                labels.append(label_code)
                
    return np.array(data), np.array(labels)

# Define the CNN architecture with data augmentation layers included
def build_cnn():
    model = models.Sequential([
        layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1)),
        
        
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.1), # Rotates up to 10%
        layers.RandomZoom(0.1),     # Zooms in/out up to 10%
        
        # --- The original CNN Architecture ---
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        
        layers.Conv2D(64, (3, 3), activation='relu', name='last_conv_layer'),
        layers.MaxPooling2D((2, 2)),
        
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.5), 
        layers.Dense(1, activation='sigmoid') 
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy']) 
    return model


print("\n--- PHASE 1: RANDOM FOREST (TESTING ALL FILTERS) ---")

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

for filter_type in FILTERS:
    X, y = load_data(DATASET_SIZE, filter_type)
    if len(X) == 0: continue
        
    # Strict 70/15/15 Split
    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=(15/85), random_state=42, stratify=y_temp)
    
    # Train and Predict 
    rf_model.fit(X_train, y_train)
    y_pred = rf_model.predict(X_test)
    
    results_log.append({
        "Model": "Random Forest",
        "Filter Type": filter_type,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, zero_division=0)
    })


print("\n--- PHASE 2: DEEP LEARNING (CNN) ---")

# Deep Learning strictly uses Raw data
X_flat, y = load_data(DATASET_SIZE, "Raw")

if len(X_flat) > 0:
    # Unflatten and Normalize (Scale 0-255 down to 0.0-1.0)
    X_cnn = X_flat.reshape(-1, IMG_SIZE, IMG_SIZE, 1) / 255.0
    
    # Strict 70/15/15 Split
    X_temp, X_test, y_temp, y_test = train_test_split(X_cnn, y, test_size=0.15, random_state=42, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=(15/85), random_state=42, stratify=y_temp)
    
    cnn_model = build_cnn()
    
    # Proper training with 20 epochs and Validation monitoring
    print("Training CNN on Raw data... (This will take a minute)")
    cnn_model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=20, batch_size=32, verbose=1)
    
    y_pred_probs = cnn_model.predict(X_test, verbose=0)
    y_pred = np.round(y_pred_probs).flatten()
    
    results_log.append({
        "Model": "CNN",
        "Filter Type": "Raw (Normalized)",
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, zero_division=0)
    })
    
    # Clear memory
    tf.keras.backend.clear_session()


print("\n=== FINAL KoS3 EXPERIMENTAL RESULTS ===")
if len(results_log) == 0:
    print("CRITICAL ERROR: No models were trained! Check your dataset folders.")
else:
    df_results = pd.DataFrame(results_log)
    
    # Reorder columns for clean reading
    df_results = df_results[["Model", "Filter Type", "Accuracy", "Precision", "Recall", "F1 Score"]]
    
    # Sort by best F1 Score at the top
    df_results = df_results.sort_values(by="F1 Score", ascending=False).reset_index(drop=True)
    display(df_results)


print("\nSaving trained models to your hard drive...")

# Save the Random Forest brain
joblib.dump(rf_model, 'kos3_random_forest.pkl')

# Save the CNN brain
cnn_model.save('kos3_cnn.keras')

print("✅ Models saved successfully! You are ready to run the GUI.")    


--- PHASE 1: RANDOM FOREST (TESTING ALL FILTERS) ---
Loading 5000 images (Raw)...
Loading 5000 images (Gaussian)...
Loading 5000 images (Median)...
Loading 5000 images (Bilateral)...
Loading 5000 images (Combined)...

--- PHASE 2: DEEP LEARNING (CNN) ---
Loading 5000 images (Raw)...
Training CNN on Raw data... (This will take a minute)
Epoch 1/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 12s 71ms/step - accuracy: 0.7489 - loss: 0.4996 - val_accuracy: 0.8600 - val_loss: 0.3064
Epoch 2/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 6s 51ms/step - accuracy: 0.8869 - loss: 0.2981 - val_accuracy: 0.8547 - val_loss: 0.3144
Epoch 3/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 8s 73ms/step - accuracy: 0.9080 - loss: 0.2656 - val_accuracy: 0.7920 - val_loss: 0.4031
Epoch 4/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 7s 63ms/step - accuracy: 0.9223 - loss: 0.2327 - val_accuracy: 0.7667 - val_loss: 0.4720
Epoch 5/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 6s 50ms/step - accuracy: 0.9363 - loss: 0.2036 - val_accuracy: 0.8427 - val_loss: 0.3469
Epoch 6/20
110/

,Model,Filter Type,Accuracy,Precision,Recall,F1 Score
0,Random Forest,Combined,0.978667,0.978667,0.978667,0.978667
1,Random Forest,Gaussian,0.978667,0.981233,0.976000,0.978610
2,Random Forest,Median,0.977333,0.978610,0.976000,0.977303
3,Random Forest,Raw,0.976000,0.981132,0.970667,0.975871
4,Random Forest,Bilateral,0.976000,0.981132,0.970667,0.975871
5,CNN,Raw (Normalized),0.809333,1.000000,0.618667,0.764415



Saving trained models to your hard drive...
✅ Models saved successfully! You are ready to run the GUI.
